# get

> Load puzzles and submit answers from SolveIt and Jupyter.

In [ ]:
#| default_exp get

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.all import *

In [ ]:
#| export
from basedpl import *
from baocd.core import EST_OFFSET, Puzzle, default_token
from baocd.utils import (
    AocdError, bstr, file_name, join_path, pystr,
)

## Get a puzzle and submit an answer

In [ ]:
#| export
_PATTERN_YEAR = r"201[5-9]|202[0-9]"
_PATTERN_DAY = r"2[0-5]|1[0-9]|[1-9]"

_aoc_est_now = atop(dict(zone=EST_OFFSET) | date.undo, time)
_recent_year = fork(same["year"], subtract, atop(less | 12, same["month"]))
_cur_day_dec = fork(same["day"], minimum, atop(less | 2025, same["year"]).agenda(Array([12, 25])))
_strip_part = replace | Array([r"part.*$", ""])
_strip_year = replace | Array([_PATTERN_YEAR, ""])
_extract_yrs = atop(unique, vfi[1][0].each, (r | _PATTERN_YEAR)["text"])
_extract_days = atop(unique, (r | _PATTERN_DAY)["text"], _strip_year, _strip_part, file_name)


def _most_recent_year():
    "This year if December (AoC EST), otherwise last year."
    yr = _recent_year(_aoc_est_now(0))
    if int(yr) < 2015: raise AocdError("Time travel not supported yet")
    return yr


def _current_day():
    "Most recent unlocked AoC day in December (EST), or `Array(1)` otherwise."
    now = _aoc_est_now(0)
    return _cur_day_dec(now) if int(now.month) == 12 else Array(1)


def _get_solveit_path():
    "Detect the active SolveIt dialog path via `dialoghelper`, or None."
    try:
        import dialoghelper
        if not (dname := dialoghelper.find_dname(required=False)): return None
        try:
            if (root := dialoghelper.data_root()): return join_path(str(root), str(dname))
        except Exception: pass
        return bstr(dname)
    except Exception:
        return None


def _extract_day_year_from_path(abspath):
    "Parse `(day, year)` from a file path via tacit `_extract_yrs` and `_extract_days`."
    yrs = _extract_yrs(abspath)
    if len(yrs) > 1: raise AocdError("Failed introspection of year")
    year = Array(int(yrs[0])) if len(yrs) else None
    days = _extract_days(abspath)
    if len(days) == 1: return Array(int(pystr(days[0]))), year
    raise AocdError("Failed introspection of day")


def get_puzzle(session=None, day=None, year=None):
    "Get a puzzle for the given date, using the SolveIt dialog or calendar for omitted values."
    if (day is None or year is None) and (dialog := _get_solveit_path()) is not None:
        try:
            dialog_day, dialog_year = _extract_day_year_from_path(dialog)
        except AocdError:
            pass
        else:
            if day is None: day = dialog_day
            if year is None: year = dialog_year
    tok = default_token() if session is None else bstr(session)
    return Puzzle(year=int(_most_recent_year() if year is None else year), day=int(_current_day() if day is None else day), token=tok)


def submit(answer, part=None, reopen=True, quiet=False, *, puzzle=None):
    "Submit an answer for a supplied puzzle or for the current SolveIt puzzle."
    if puzzle is None: puzzle = get_puzzle()
    return puzzle.submit(answer, part=part, reopen=reopen, quiet=quiet)

In [ ]:
d, y = _extract_day_year_from_path(Array("/app/data/bpl/aoc/aoc_2025/day1.ipynb"))
test_eq((int(d), int(y)), (1, 2025))
d2, y2 = _extract_day_year_from_path(Array("/home/u/xmas_problem_2016_25b_part2.py"))
test_eq((int(d2), int(y2)), (25, 2016))
d3, y3 = _extract_day_year_from_path(Array("/tmp/q03.py"))
test_eq(int(d3), 3); test_is(y3, None)
test_eq(int(_most_recent_year()) >= 2025, True)
test_eq(1 <= int(_current_day()) <= 25, True)

from unittest.mock import Mock, patch

with patch.dict(get_puzzle.__globals__, {"_get_solveit_path": lambda: Array("/aoc_2025/day1.ipynb")}):
    inferred = get_puzzle(session="test-token")
    explicit = get_puzzle(session="test-token", day=2, year=2024)
    test_eq((inferred.day, inferred.year), (1, 2025))
    test_eq((explicit.day, explicit.year), (2, 2024))

target = Mock()
test_is(submit(42, part="b", reopen=False, quiet=True, puzzle=target), target.submit.return_value)
target.submit.assert_called_once_with(42, part="b", reopen=False, quiet=True)